In [0]:
import time
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F

In [0]:
%run ../00_utils/common

In [0]:
S3_BUCKET = dbutils.secrets.get(scope="retailmax", key="s3_bucket")

In [0]:
TABLES = {
    "mstr_proveedores": "id_proveedor",
    "mstr_articulos": "art_id",
    "mstr_tiendas": "id_tienda",
    "crm_miembros": "id_miembro",
    "trans_ventas": "id_trans",
    "inv_stock_diario": "id_snapshot",
    "post_devoluciones": "id_devolucion",
}


In [0]:
def get_last_ingested(table: str) -> str:
    """
    Retorna el timestamp de la última ingesta registrada en pipeline_log.
    Si no existe registro previo retorna None.
    """
    try:
        row = (spark.table(f"{CATALOG}.bronze.pipeline_log")
               .filter(
                   (F.col("layer") == "bronze") &
                   (F.col("table_name") == table) &
                   (F.col("status") == "OK")
                )
               .orderBy(F.col("ts_log").desc())
               .limit(1)
               .collect())
        return row[0]["ts_log"] if row else None
    except Exception:
        return None


In [0]:
def get_recent_pks(table: str, pk: str, days_back: int = 7):
    """
    En vez de escanear toda bronze, escanea solo
    las últimas N particiones para obtener PKs recientes.
    N debe cubrir la ventana de posibles duplicados de la fuente.
    """
    cutoff = datetime.now() - timedelta(days=days_back)
    return (spark.table(f"{CATALOG}.bronze.{table}")
            .filter(
                (F.col("_ingest_year") >= cutoff.year) &
                (F.col("_ingest_month") >= cutoff.month) &
                (F.col("_ingest_day") >= cutoff.day))
            .select(pk))

In [0]:
def ingest_table(table: str, pk: str, batch: str):
    s3_path = f"s3://{S3_BUCKET}/raw/postgres/{table}/{table}.parquet"
    t0 = time.time()

    try:
        df_raw = spark.read.parquet(s3_path)

        # Ingesta incremental
        last_ts = get_last_ingested(table)
        if last_ts:
            # Filtrar solo registros nuevos usando la PK contra recientes PKS
            df_recent_pks = get_recent_pks(table, pk, days_back=7)
            df_raw = df_raw.join(df_recent_pks, on=pk, how="left_anti")
            print(f"  Modo incremental desde {last_ts} — {df_raw.count()} registros nuevos")
        else:
            print(f"  Historical Backfill — {df_raw.count()} registros")

        if df_raw.isEmpty():
            log_run("bronze", table, 0, 0, time.time() - t0, batch=batch)
            print(f"  Sin registros nuevos. Saltando.")
            return

        # Metadatos
        ts_now = now_utc()
        df = (df_raw
              .withColumn("_ingested_at", F.lit(ts_now))
              .withColumn("_source_system", F.lit("postgres"))
              .withColumn("_batch_id", F.lit(batch))
              # Partición: año/mes/día de ingesta
              .withColumn("_ingest_year", F.lit(datetime.now().year))
              .withColumn("_ingest_month", F.lit(datetime.now().month))
              .withColumn("_ingest_day", F.lit(datetime.now().day)))

        # Escritura particionada
        target = f"{CATALOG}.bronze.{table}"
        (df.write
           .format("delta")
           .mode("append")
           .partitionBy("_ingest_year", "_ingest_month", "_ingest_day")
           .saveAsTable(target))

        n_records  = df.count()
        size_bytes = spark.sql(
            f"DESCRIBE DETAIL {target}"
        ).select("sizeInBytes").collect()[0][0] or 0

        log_run("bronze", table, n_records, size_bytes, time.time() - t0, batch=batch)
        print(f"  [DONE] {table}: {n_records:,} registros en {time.time()-t0:.1f}s")

    except Exception as e:
        log_run("bronze", table, 0, 0, time.time() - t0, status="ERROR", error=str(e), batch=batch)
        print(f"  [ERROR] {table}: {e}")


In [0]:
batch = batch_id()
print(f"Bronze Ingesta | batch={batch}")
for table, pk in TABLES.items():
    ingest_table(table, pk, batch)
print(" Bronze completado.")